# 🌐 Notebook 4: Distributed Limits & Good-Citizen Clients

Real services run on **many servers behind a load balancer**. An
in-memory limiter on each server is *per-server*, so the total allowed
rate is `N_servers × limit`. That's rarely what you want.

We'll cover two production topics:

1. **Server side** — how a shared counter (usually **Redis**) makes the
   limit global across servers.
2. **Client side** — how a *well-behaved* client reacts to `429` with
   **exponential backoff + jitter** (critical to avoid the "thundering
   herd" retry storm).

The notebook runs entirely in-memory (no Redis needed) using a fake
shared store, so the concepts are the focus.


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## ❌ Bad: each server has its own limiter

If 3 servers each enforce "100 req/sec", clients actually get 300 req/sec
total — and the number silently changes when you scale up or down. Not fun
to reason about.


In [ ]:
import time, random, threading
from collections import defaultdict, deque

random.seed(7)   # deterministic output

class LocalLimiter:
    def __init__(self, limit, window):
        self.limit, self.window = limit, window
        self.events = defaultdict(deque)
    def allow(self, key):
        now = time.monotonic()
        q = self.events[key]
        while q and q[0] < now - self.window:
            q.popleft()
        if len(q) < self.limit:
            q.append(now)
            return True
        return False

# 3 independent servers, each enforcing its own 10/s limiter
servers = [LocalLimiter(10, 1.0) for _ in range(3)]
ok = sum(random.choice(servers).allow('alice') for _ in range(60))
print(f'alice got {ok} allowed in 1s — intended limit was 10/s, actual ceiling is ~30/s')
print('The limit silently changes every time you scale the fleet up or down.')

## ❌ Still bad: a shared counter read with two round trips

"Just put the counter in Redis" is the obvious fix, and the obvious implementation is wrong:

```python
count = redis.get(key)          # round trip 1
if count < limit:
    redis.set(key, count + 1)   # round trip 2
    return True
```

Between those two round trips, every other server is doing the same thing. They all read the same value, they all decide they're under the limit, and they all write back the *same* incremented number — so most of the increments vanish (a classic **lost update**).

This is not a rare interleaving you have to get unlucky to hit. Under real concurrency it is the *normal* case, because the gap between the two calls is a network round trip — an eternity.

In [ ]:
class FakeRedis:
    """A shared store with a realistic 2ms round trip per command."""
    def __init__(self):
        self.data = {}
        self._lock = threading.Lock()   # models Redis being single-threaded per command

    def _round_trip(self):
        time.sleep(0.002)               # the network hop — this is where the race lives

    def get(self, key):
        self._round_trip()
        with self._lock:
            return self.data.get(key, 0)

    def set(self, key, value):
        self._round_trip()
        with self._lock:
            self.data[key] = value

    def incr_if_below(self, key, limit):
        """ONE round trip; the check and the increment happen together, server-side.
        This is what a Redis Lua script (or INCR + compare) buys you."""
        self._round_trip()
        with self._lock:
            current = self.data.get(key, 0)
            if current < limit:
                self.data[key] = current + 1
                return True
            return False


def racy_allow(redis, key, limit):
    count = redis.get(key)              # round trip 1  ─┐
    if count < limit:                   #                │ another server slips in here
        redis.set(key, count + 1)       # round trip 2  ─┘
        return True
    return False

def atomic_allow(redis, key, limit):
    return redis.incr_if_below(key, limit)


def hammer(allow_fn, concurrent=60, limit=10):
    """`concurrent` requests land at the same instant, spread across the fleet."""
    redis, results, lock = FakeRedis(), [], threading.Lock()
    def one_request():
        ok = allow_fn(redis, 'alice', limit)
        with lock:
            results.append(ok)
    threads = [threading.Thread(target=one_request) for _ in range(concurrent)]
    for t in threads: t.start()
    for t in threads: t.join()
    return sum(results), redis.data.get('alice', 0)

allowed, counter = hammer(racy_allow)
print(f'read-then-write: {allowed}/60 allowed against a limit of 10; '
      f'counter ended at {counter}')
print(f'-> the limiter let through {allowed / 10:.0f}x its limit, and {60 - counter} of the')
print('   increments were lost entirely. The counter is not just late, it is wrong.')

## ✅ Good: make the whole decision atomic

The fix is not a faster network or a lock in your application — it's moving the **entire decision** to the store, so it happens in one indivisible step. In Redis that's a Lua script (or `INCR` and read the reply, which is atomic by itself):

```lua
-- KEYS[1]=key, ARGV={now, window, limit, unique_member}
redis.call('ZREMRANGEBYSCORE', KEYS[1], 0, ARGV[1] - ARGV[2])
local count = redis.call('ZCARD', KEYS[1])
if count < tonumber(ARGV[3]) then
  redis.call('ZADD', KEYS[1], ARGV[1], ARGV[4])   -- unique member
  redis.call('EXPIRE', KEYS[1], ARGV[2])
  return 1   -- allowed
else
  return 0   -- denied
end
```

> 💡 **Three subtleties the script encodes:**
> 1. **One round trip.** Check and add are the same operation — that's the whole point.
> 2. **Check before adding** — otherwise rejected requests still inflate the window and starve the legitimate ones behind them.
> 3. **Unique member** (e.g. `f"{now}-{uuid4()}"`) — two requests in the same millisecond produce identical scores, and a sorted set would silently keep only one.

Same 60 concurrent requests, same store, one round trip instead of two:

In [ ]:
allowed, counter = hammer(atomic_allow)
print(f'atomic check-and-incr: {allowed}/60 allowed against a limit of 10; '
      f'counter ended at {counter}  ✅')

### The real thing: an atomic sliding-window log, with real 429 responses

The toy counter above never expires. Here's the same atomicity applied to the sliding-window log from notebook 3 — and this time the limiter returns an actual HTTP response, because a bare `False` is useless to the client.

In [ ]:
class SharedStore:
    """Fake Redis with a sliding-window log — the Lua script above, in Python."""
    def __init__(self):
        self._data = defaultdict(list)
        self._lock = threading.Lock()

    def check_and_add(self, key, now, window, limit):
        """Atomically: drop expired, check the limit, add if under. -> (allowed, count)"""
        with self._lock:
            arr = self._data[key]
            cutoff = now - window
            i = 0
            while i < len(arr) and arr[i] < cutoff:
                i += 1
            if i:
                del arr[:i]
            if len(arr) < limit:
                arr.append(now)      # record ALLOWED events only
                return True, len(arr)
            return False, len(arr)

    def retry_after(self, key, now, window):
        """When does the oldest event fall out of the window? -> the Retry-After value."""
        with self._lock:
            arr = self._data[key]
            return max(0.0, (arr[0] + window) - now) if arr else 0.0

class DistributedLimiter:
    def __init__(self, store, limit, window):
        self.store, self.limit, self.window = store, limit, window

    def handle(self, key):
        """Returns a real HTTP response, not just a bool — the client needs the headers."""
        now = time.monotonic()
        allowed, count = self.store.check_and_add(key, now, self.window, self.limit)
        headers = {
            'X-RateLimit-Limit': str(self.limit),
            'X-RateLimit-Remaining': str(max(0, self.limit - count)),
        }
        if allowed:
            return {'status': 200, 'headers': headers}
        retry = self.store.retry_after(key, now, self.window)
        headers['Retry-After'] = f'{retry:.2f}'      # seconds; integers in real HTTP
        return {'status': 429, 'headers': headers}

store = SharedStore()
servers = [DistributedLimiter(store, limit=10, window=1.0) for _ in range(3)]

responses = [random.choice(servers).handle('alice') for _ in range(60)]
ok = sum(r['status'] == 200 for r in responses)
print(f'alice got {ok} allowed in 1s across 3 servers — matches the global limit ✅')
print('a 429     :', responses[ok])

# Retry-After is computed, not guessed: it counts down as the window ages out.
for pause in (0.3, 0.3):
    time.sleep(pause)
    print(f'after {pause}s more:', servers[0].handle('alice')['headers']['Retry-After'], 's')

### ⚖️ What the shared store costs you

A global limiter is not free, and the trade-offs are the interesting part of the design review:

- **A network hop on every request.** ~1 ms added to *every* call, including the 99.99% you were going to allow anyway.
- **A new hard dependency.** Redis is now in the critical path of every request. See notebook 5 on fail-open vs fail-closed — you must pick one *before* the outage.
- **A hot key.** All traffic for one tenant hits one Redis slot. Big tenants can saturate a single node; shard the key (`alice:{shard}`) and divide the limit if that happens.
- **Clock skew.** If servers pass their own `now` to the store, a server whose clock is 2 s fast can evict the window early. Use the **store's** clock (`redis.call('TIME')`) as the single source of truth.

The common production compromise is **two tiers**: a generous per-server local limiter that rejects the obvious flood without a network hop, and the shared store behind it for the exact global number. You pay for Redis only on traffic that got past the cheap check.

## 🧑‍💻 Being a good-citizen client

When the server replies `429 Too Many Requests`, the **wrong** reaction is:
```
while not ok: retry_immediately()
```
Every client does that simultaneously → the server gets a bigger spike the
moment it recovers. This is called a **thundering herd**.

The right reaction is **exponential backoff + jitter**:
- wait `base * 2^attempt` seconds, but
- add a random offset so clients don't all retry in lock-step.

AWS's canonical paper: *"Exponential Backoff and Jitter"* recommends
**"full jitter"** — `sleep = random(0, base * 2^attempt)`.


In [ ]:
import random

def call_with_backoff(do_call, max_attempts=6, base=0.1, cap=5.0):
    """do_call() returns (ok: bool, retry_after: float|None)."""
    for attempt in range(max_attempts):
        ok, retry_after = do_call()
        if ok:
            return True, attempt
        if retry_after is not None:
            # Server told us how long to wait — respect it.
            time.sleep(retry_after)
        else:
            # Full jitter: uniform in [0, min(cap, base * 2**attempt)]
            wait = random.uniform(0, min(cap, base * (2 ** attempt)))
            time.sleep(wait)
    return False, max_attempts

# Fake server: fails for first 3 calls, then succeeds.
calls = {'n': 0}
def flaky():
    calls['n'] += 1
    if calls['n'] <= 3:
        return False, 0.05   # 429 with Retry-After: 50ms
    return True, None

ok, attempts = call_with_backoff(flaky)
print(f"success={ok} after {attempts+1} attempts")

## 💥 Why jitter matters (mini simulation)

Imagine 500 clients all hit a rate-limited endpoint at `t=0` and all get
`429`. If they **all** retry at `t=1s`, you get another 500-request spike.
With jitter, retries *spread out* across the interval.


In [ ]:
import matplotlib.pyplot as plt

def synchronized_retry_times(n=500, base=1.0):
    return [base] * n   # everyone waits exactly base seconds

def jittered_retry_times(n=500, base=1.0):
    return [random.uniform(0, base) for _ in range(n)]

sync_t = synchronized_retry_times()
jit_t  = jittered_retry_times()

fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
axes[0].hist(sync_t, bins=20, color='red')
axes[0].set_title('no jitter — thundering herd')
axes[0].set_xlabel('retry time (s)'); axes[0].set_ylabel('clients')
axes[1].hist(jit_t, bins=20, color='green')
axes[1].set_title('full jitter — load spread out')
axes[1].set_xlabel('retry time (s)')
plt.tight_layout(); plt.show()

## 🧠 Takeaways

- **Per-server limits lie.** With N servers your real limit is N × the configured one, and it changes silently every time you autoscale.
- **A shared counter is not enough — the *decision* has to be atomic.** `get` then `set` is two round trips, and under concurrency almost every increment is lost. Push the whole check-and-increment into the store (Lua script, `INCR`, or `redis-cell`).
- **Tell clients what to do.** `429` + `Retry-After` + `X-RateLimit-*`. A rejection with no guidance just becomes a tight retry loop.
- **Respect `Retry-After`** on the client; if it's absent, exponential backoff with **full jitter**.
- **Cap retries** — after `max_attempts`, surface a real error rather than retrying forever.
- **Know what the shared store costs**: a hop per request, a new hard dependency, hot keys, and clock skew. Decide fail-open vs fail-closed *before* Redis goes down (notebook 5).

### Further reading
- AWS blog — *"Exponential Backoff And Jitter"*
- Cloudflare — *"How we built rate limiting capable of scaling to millions of domains"*
- GitHub REST API — *"Resources in the REST API > Rate limiting"* (headers)
- Stripe API docs — *"Rate limits"* (token bucket + 429 behavior)